# Chapter 35: Hidden Markov Models

## Learning Objectives
- Model latent regimes with Markov transitions
- Compute sequence likelihood via Forward algorithm
- Decode most probable state path via Viterbi
- Interpret sequence-level probabilistic outputs

## Prerequisites
- Strong understanding of chapters 0-29
- Python and basic linear algebra
- Numpy for probabilistic/deep chapters


# Chapter 35: Hidden Markov Models (HMM)

## Why this chapter matters
HMMs are classic probabilistic sequence models used in speech, bioinformatics, forecasting states, and anomaly pipelines where latent regime modeling is useful.

## Learning goals
1. Understand latent state transition and emission structure.
2. Implement Forward algorithm for sequence likelihood.
3. Implement Viterbi decoding for most likely state path.
4. Interpret model outputs for regime segmentation.

## Model components
- hidden states: \(z_t\)
- transition matrix: \(A_{ij}=P(z_t=j|z_{t-1}=i)\)
- emission matrix: \(B_{j,o}=P(x_t=o|z_t=j)\)
- initial distribution: \(\pi_i=P(z_0=i)\)

## Forward algorithm
Dynamic programming recurrence:
\[
\alpha_t(j) = B_{j, x_t}\sum_i \alpha_{t-1}(i)A_{ij}
\]
Sequence likelihood:
\[
P(x_{1:T}) = \sum_j \alpha_T(j)
\]

## Viterbi decoding
Replace sum with max and track backpointers:
\[
\delta_t(j) = B_{j,x_t}\max_i \delta_{t-1}(i)A_{ij}
\]
Backtrack to recover most likely hidden state path.

## Industry usage tips
- Work in log-space for long sequences to avoid underflow.
- Validate emissions and transitions with domain constraints.
- Compare decoded regimes to known events.

## Assignment
1. Implement log-space forward and Viterbi.
2. Decode latent states for synthetic regime-switching sequence.
3. Build confusion table vs known hidden regimes.


In [ ]:
from __future__ import annotations

from typing import List, Tuple


def forward_algorithm(pi, A, B, obs: List[int]) -> float:
    n_states = len(pi)
    alpha = [pi[s] * B[s][obs[0]] for s in range(n_states)]

    for t in range(1, len(obs)):
        new_alpha = [0.0] * n_states
        for j in range(n_states):
            s = 0.0
            for i in range(n_states):
                s += alpha[i] * A[i][j]
            new_alpha[j] = s * B[j][obs[t]]
        alpha = new_alpha

    return sum(alpha)


def viterbi_decode(pi, A, B, obs: List[int]) -> Tuple[float, List[int]]:
    n_states = len(pi)
    T = len(obs)

    dp = [[0.0] * n_states for _ in range(T)]
    bp = [[0] * n_states for _ in range(T)]

    for s in range(n_states):
        dp[0][s] = pi[s] * B[s][obs[0]]

    for t in range(1, T):
        for j in range(n_states):
            best_prob, best_state = -1.0, 0
            for i in range(n_states):
                p = dp[t - 1][i] * A[i][j]
                if p > best_prob:
                    best_prob = p
                    best_state = i
            dp[t][j] = best_prob * B[j][obs[t]]
            bp[t][j] = best_state

    last_state = max(range(n_states), key=lambda s: dp[T - 1][s])
    best_prob = dp[T - 1][last_state]

    path = [last_state]
    for t in range(T - 1, 0, -1):
        path.append(bp[t][path[-1]])
    path.reverse()

    return best_prob, path


if __name__ == "__main__":
    # 0=Rainy, 1=Sunny
    pi = [0.6, 0.4]
    A = [
        [0.7, 0.3],
        [0.4, 0.6],
    ]
    # emissions: 0=walk,1=shop,2=clean
    B = [
        [0.1, 0.4, 0.5],
        [0.6, 0.3, 0.1],
    ]

    obs = [0, 1, 2, 2, 1, 0]

    likelihood = forward_algorithm(pi, A, B, obs)
    best_prob, path = viterbi_decode(pi, A, B, obs)

    print("Sequence likelihood:", round(likelihood, 8))
    print("Viterbi probability:", round(best_prob, 8))
    print("Best hidden path:", path)


## Checkpoint

1. You can calculate forward probabilities recursively.
2. You can decode hidden path with dynamic programming.
3. You can map states to interpretable regimes.

## Guided Exercise
Create a second observation sequence and compare likelihood under the same HMM parameters.

In [ ]:
# TODO (Student Exercise)
obs2 = [2,2,1,0,0]
print('Likelihood obs2:', round(forward_algorithm(pi, A, B, obs2), 8))
print('Viterbi obs2:', viterbi_decode(pi, A, B, obs2))

## Quick Quiz

**Q1.** What is the difference between forward and Viterbi outputs?

**Answer:** Forward gives total sequence likelihood; Viterbi gives best single path.

**Q2.** Why use dynamic programming here?

**Answer:** It avoids exponential enumeration of all state sequences.

**Q3.** When should log-space be used?

**Answer:** For long sequences to prevent numerical underflow.


## Homework
Implement log-space Viterbi and compare numerical stability on long sequences.